# Exploring and preprocessing datasets

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Step 1: Load and Prepare Dataset

In this step, multiple data sources were loaded and integrated into a unified dataset for analysis. The datasets include user information, transaction history, and churn labels. The integration was performed using the user identifier (msno) to enable a comprehensive view of each user's behavior.

In [2]:
train = pd.read_csv("../data/train_v2.csv")
members = pd.read_csv("../data/members_v3.csv")
transactions = pd.read_csv("../data/transactions_v2.csv")

print("Train:", train.shape)
print("Members:", members.shape)
print("Transactions:", transactions.shape)

Train: (970960, 2)
Members: (6769473, 6)
Transactions: (1431009, 9)


## STEP 2 — Target Analysis
The distribution of the target variable (is_churn) was examined to understand the class balance. This step is crucial for identifying potential bias in the dataset and determining appropriate modeling strategies.

In [3]:
train['is_churn'].value_counts()
train['is_churn'].value_counts(normalize=True)

is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64

- Data is Imbalanced: “~9% churn → imbalanced dataset”
- “The dataset is highly imbalanced, with only a small proportion of churned users.”

## 3. MERGE RAW DATA (EDA ONLY)

In [4]:
df_raw = train.merge(members, on='msno', how='left')
df_raw = df_raw.merge(transactions, on='msno', how='left')

print(df_raw.shape)
df_raw.sample(15)

(1169418, 15)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
589966,AmhdTfEdhftuDCTmGI1WkYAKMPq8qhUgD0O1442GgrA=,0,13.0,21.0,female,9.0,20140209.0,39.0,30.0,149.0,149.0,1.0,20170331.0,20170527.0,0.0
460615,TVSRUEfhvfVT6MU9yoDe+X58rhPgw0ltzRtlJUA4GUc=,0,1.0,0.0,NaN,7.0,20111121.0,41.0,30.0,149.0,149.0,1.0,20170311.0,20170411.0,0.0
196349,z8djI+51SfH03dLTetLYrsTFDVh8F4uAzD/qZTUQBhk=,0,1.0,0.0,NaN,4.0,20160324.0,36.0,30.0,180.0,180.0,0.0,20170306.0,20170405.0,0.0
654963,q7V/0fSdkl4e3jxFTHpMDiXiSHgN6NeAEX0+GBtKHmg=,0,15.0,24.0,male,9.0,20060415.0,39.0,30.0,149.0,149.0,1.0,20170331.0,20170512.0,0.0
188498,lr+ax6bTjauMV9qFCIlTLBjQUiNEW4xyDFPSZto16Z8=,0,1.0,0.0,NaN,7.0,20121026.0,41.0,30.0,99.0,99.0,1.0,20170307.0,20170407.0,0.0
469782,Y8Gad+gisHLPLMDerm7m/iaf70qmdJu96KIIA13eQts=,0,22.0,42.0,female,9.0,20051125.0,19.0,30.0,149.0,149.0,1.0,20170331.0,20170430.0,0.0
908704,Q5H5TEC2sssL+Weg5g00U79f2DuAgrU9CSuVG7r+Bko=,0,1.0,0.0,NaN,7.0,20111228.0,41.0,30.0,149.0,149.0,1.0,20170309.0,20170409.0,0.0
103622,RzDQfm7aVMNwxanYy7A1PJUfGoloYBProbZSTX7TjPQ=,0,1.0,0.0,NaN,4.0,20151020.0,36.0,30.0,180.0,180.0,0.0,20170310.0,20170409.0,0.0
423127,INp/3KTzwCE0j18L7uHw9ph4CGmrg6+lAYlAfv7MVdA=,0,1.0,0.0,NaN,7.0,20160630.0,41.0,30.0,99.0,99.0,1.0,20170331.0,20170430.0,0.0
667502,M2/tlKl8R7ilBmpyPuxOxL3hDsvLCkUqpTio3aAdCMY=,0,1.0,0.0,NaN,7.0,20131102.0,41.0,30.0,99.0,99.0,1.0,20170331.0,20170430.0,0.0


## Step 4 — Data Quality Assessment
Data quality checks were conducted to identify missing values, inconsistencies, and outliers in the dataset. Ensuring data quality is essential for building reliable machine learning models.

In [5]:
df_raw.isnull().sum()

msno                           0
is_churn                       0
city                      116836
bd                        116836
gender                    668413
registered_via            116836
registration_init_time    116836
payment_method_id          37382
payment_plan_days          37382
plan_list_price            37382
actual_amount_paid         37382
is_auto_renew              37382
transaction_date           37382
membership_expire_date     37382
is_cancel                  37382
dtype: int64

- Incorrect data
- “Age contains unrealistic values”

## Step 5 — Demographic Feature Analysis
Demographic features such as age, gender, and city were analyzed to determine their relationship with churn behavior. Group-based statistics were used to compare churn rates across different demographic segments.

In [6]:
# Age vs churn
df_raw.groupby('is_churn')['bd'].mean()

is_churn
0    14.009617
1    16.728249
Name: bd, dtype: float64

The churn tends to differ with age.

In [7]:
# City vs churn
df_raw.groupby('city')['is_churn'].mean().sort_values(ascending=False)

city
21.0    0.173373
8.0     0.167883
12.0    0.165597
4.0     0.159406
5.0     0.158681
3.0     0.153846
14.0    0.152156
10.0    0.152148
19.0    0.151899
9.0     0.151625
16.0    0.148444
6.0     0.146283
13.0    0.146242
22.0    0.142301
15.0    0.141933
18.0    0.129561
11.0    0.128976
20.0    0.121979
7.0     0.121882
17.0    0.121701
1.0     0.102598
Name: is_churn, dtype: float64

In [8]:
# Gender vs churn
df_raw.groupby('gender')['is_churn'].mean()

gender
female    0.151336
male      0.151975
Name: is_churn, dtype: float64

## Step 6 — Transaction Behavior Analysis
Transaction-related features were analyzed to understand user payment behavior. Key indicators such as cancellation frequency, payment amount, and auto-renewal status were examined in relation to churn.

In [9]:
# Cancel
df_raw.groupby('is_churn')['is_cancel'].mean()

is_churn
0    0.013052
1    0.172488
Name: is_cancel, dtype: float64

- 1 is cancel
- “Users who cancel more frequently are more likely to churn”

In [10]:
# Auto renew
df_raw.groupby('is_churn')['is_auto_renew'].mean()

is_churn
0    0.940522
1    0.699101
Name: is_auto_renew, dtype: float64

- 1 is not auto renew → churn

In [11]:
# Payment
df_raw.groupby('is_churn')['actual_amount_paid'].mean()

is_churn
0    131.213343
1    268.869617
Name: actual_amount_paid, dtype: float64

- Low spending → higher churn

In [12]:
# Day left — compute days_left directly on df_raw
df_raw['membership_expire_date_dt'] = pd.to_datetime(
    df_raw['membership_expire_date'].astype(str).str.split('.').str[0],
    format='%Y%m%d',
    errors='coerce'
)
reference_date = pd.to_datetime('2017-03-01')
df_raw['days_left'] = (df_raw['membership_expire_date_dt'] - reference_date).dt.days

df_raw.groupby('is_churn')['days_left'].mean()

is_churn
0     61.487698
1    136.005151
Name: days_left, dtype: float64

## STEP 7 — Temporal Analysis
Time-based features were analyzed to capture temporal patterns in user behavior. Features such as subscription duration and time remaining until expiration were derived and evaluated.

In [13]:
# Convert datetime
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'].astype(str),
    format='%Y%m%d'
)

transactions['membership_expire_date'] = pd.to_datetime(
    transactions['membership_expire_date'].astype(str),
    format='%Y%m%d'
)
transactions[['transaction_date', 'membership_expire_date']].head()

last_trans = transactions.sort_values('membership_expire_date') \
                         .groupby('msno') \
                         .tail(1)

df = train.merge(last_trans, on='msno', how='left')

reference_date = pd.to_datetime("2017-03-01")

df['days_left'] = (
    df['membership_expire_date'] - reference_date
).dt.days

df.groupby('is_churn')['days_left'].mean()

is_churn
0     50.708947
1    122.588652
Name: days_left, dtype: float64